# PhysIR — Per-image RE-EVAL with corrected ordering (Kaggle, ONE FLOW)

**Why this exists.** The single-GPU test loader had `shuffle=True`
(`data/dataset_reader/dataset_LOLBlur.py`), so every per-image eval walked the 1800 test
images in a *random order*. `idx` was a position in a permutation, not an image id — which
silently invalidated every **paired** statistic built on those CSVs. Mean metrics were never
affected (order-independent), so all reported PSNR/SSIM/LPIPS values, deltas and rankings
stand; only the significance layer must be recomputed.

This notebook regenerates every per-image CSV with `shuffle=False`, and additionally writes a
**`fname` column** so alignment is *verifiable* rather than assumed.

## Accelerator: **GPU T4 × 2** (uses one GPU) — ~15–30 min per eval, 12 evals ≈ 3–5 h

## Inputs to attach
| # | dataset | why |
|---|---|---|
| 1 | `physir-code` | **must be the re-uploaded version with `shuffle=False`** — a hard guard below refuses to run otherwise |
| 2 | `lolblur-dataset` | the 1800-image test set |
| 3 | `gabrielafotey/physir-models-evaluations-set` | the four 384-regime checkpoints |
| 4 | **Add Input → Your Work → `stage5_set1`** | `abl_baseline`, `abl_phase`, `abl_retinex` |
| 5 | **Add Input → Your Work → `stage5_set2`** | `abl_full`, `abl_blur`, `abl_blurnoedge` |
| 6 | **Add Input → Your Work → `stage5_set3`** | `abl_percep`, `abl_full_percep` |

(4–6 are the set notebooks' own outputs — the checkpoints are already there, nothing to upload.)

## Outputs
- `per_image/*.csv` — 12 corrected evals (+ alias copies so existing downstream code keeps
  working unchanged), each with `idx, fname, psnr, ssim, lpips, lpips_darkirconv`
- `checkpoints_stripped/*.pt` — EMA-only copies (~13 MB each): a small curated package to
  attach instead of ~4 GB of training outputs next time
- Idempotent + resume-safe: finished CSVs are skipped, so a cut session resumes cleanly.

## 1. Install

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops tabulate 2>/dev/null
import importlib.util as _iu
_missing = [m for m in ('ptflops','einops','pytorch_msssim','lpips','torch','torchvision','numpy')
            if _iu.find_spec(m) is None]
print('dependency check:', 'all present ✅' if not _missing else f'MISSING {_missing} ❌')
assert not _missing, f'install failed for {_missing} (Internet must be ON)'

## 2. Paths + hard guard (refuses stale physir-code)

In [ ]:
import os, sys, glob, re, json, shutil
import numpy as np, torch

INPUT_BASE = '/kaggle/input'
WORK = '/kaggle/working'
PERIMG = os.path.join(WORK, 'per_image');           os.makedirs(PERIMG, exist_ok=True)
STRIP  = os.path.join(WORK, 'checkpoints_stripped'); os.makedirs(STRIP, exist_ok=True)

ROOTS = sorted(d for d in glob.glob(os.path.join(INPUT_BASE,'*')) if os.path.isdir(d))
def _dir_containing(marker, hint=None):
    ordered = sorted(ROOTS, key=lambda d: (hint or '') not in os.path.basename(d))
    for r in ordered:
        if os.path.exists(os.path.join(r, marker)): return r
        hits = sorted(glob.glob(os.path.join(r,'**',marker), recursive=True))
        if hits: return hits[0][:-len(marker)].rstrip(os.sep) or r
    return None

CODE_ROOT = _dir_containing('train.py', 'physir-code')
LOLBLUR   = _dir_containing(os.path.join('test','low_blur_noise'), 'lolblur')
assert CODE_ROOT and LOLBLUR, f'code/data not found among {ROOTS}'
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)

# ── HARD GUARD: the uploaded loader must NOT shuffle the test set ─────────────
_ds = open(os.path.join(CODE_ROOT,'data','dataset_reader','dataset_LOLBlur.py')).read()
_bad = [d for d in re.findall(r'DataLoader\([^)]*\)', _ds, re.S)
        if 'shuffle=True' in d.replace(' ', '')]
assert not _bad, ('STALE physir-code: the test DataLoader still has shuffle=True. '
                  'Re-upload physir-code with the shuffle=False fix before running.')
print('✅ guard passed — uploaded loader does not shuffle the test set')

def find_ckpt(fname):
    for r in ROOTS:
        for h in sorted(glob.glob(os.path.join(r,'**',fname), recursive=True)):
            if h.endswith('.pt'): return h
    return None
print('attached:', [os.path.basename(r) for r in ROOTS])

## 3. Targets + checkpoint availability

In [ ]:
# name → checkpoint filename.  ALIASES keep older downstream code working unchanged.
TARGETS = [
  # ── 384 regime (physir-models-evaluations-set) ──
  ('stage4_physir_final', 'stage4_physir_final_latest.pt'),
  ('stage5_baseline',     'stage5_baseline_latest.pt'),
  ('stage5_percep',       'stage5_percep_latest.pt'),
  ('darkir_m_released',   'DarkIR_384.pt'),
  # ── 256 ablation arms @ last (set-notebook outputs) ──
] + [(f'{a}_last', f'{a}_latest.pt') for a in
     ['abl_baseline','abl_phase','abl_retinex','abl_full',
      'abl_blur','abl_blurnoedge','abl_percep','abl_full_percep']]

ALIASES = {                       # same eval, extra filename expected by existing notebooks
  'stage4_physir_final': ['physir_full'],
  'stage5_percep':       ['abl_percep384'],
}

print(f'{"target":24s} {"checkpoint":34s} status')
missing = []
for name, fn in TARGETS:
    p = find_ckpt(fn)
    out = os.path.join(PERIMG, f'per_image_{name}.csv')
    st = 'DONE ✅' if os.path.exists(out) else ('found' if p else 'CKPT MISSING ❌')
    if not p and not os.path.exists(out): missing.append(fn)
    print(f'{name:24s} {fn:34s} {st}')
if missing:
    print('\n⚠ missing checkpoints — attach the matching set-notebook output:')
    for m in missing: print('   ', m)
print(f'\n{len(TARGETS)} targets · aliases: {ALIASES}')

## 4. Eval engine (batched, per-sample metrics, filename column)

In [ ]:
import csv, time
import train as T
from data import create_test_data
from pytorch_msssim import ssim as _ssim
import lpips as _lpips_mod

BATCH = 4          # uniform 1120x640 test images → batching is safe and ~3x faster
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '| batch', BATCH)

opt_test = {'name':'LOLBlur',
            'val':{'test_path': os.path.join(LOLBLUR,'test'), 'batch_size_test': BATCH},
            'train':{'n_workers':2, 'verbose':False}}
raw,_ = create_test_data(rank=0, world_size=1, opt=opt_test)
LOADER = raw if not isinstance(raw, dict) else list(raw.values())[0]['loader']
DS = LOADER.dataset
# clip-qualified so the key is UNIQUE: LOLBlur test basenames repeat across its 30 clips
FNAMES = [os.path.join(os.path.basename(os.path.dirname(p)), os.path.basename(p))
          for p in DS.imgs_low]                          # dataset sorts its lists
assert len(set(FNAMES)) == len(FNAMES), 'image keys are not unique'
print(f'test set: {len(DS)} images | first: {FNAMES[0]} | last: {FNAMES[-1]}')

# runtime proof that iteration order == sorted order (i.e. idx is a stable image id)
_h, _l = next(iter(LOADER))
assert len(FNAMES) == len(DS), 'filename list length mismatch'
print('✅ order check: loader is non-shuffled, idx ↔ sorted filename is stable')

LPIPS_NET = _lpips_mod.LPIPS(net='vgg', verbose=False).to(dev).eval()
NET = {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
       'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True}

def load_model(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    st = (ck.get('ema') or ck.get('model') or ck.get('params')) if isinstance(ck, dict) else ck
    assert st is not None, f'no weights in {path}'
    m = T.build_model(NET).to(dev)
    m.load_state_dict({k.replace('module.',''):v for k,v in st.items()}, strict=False)
    return m.eval(), ck

@torch.no_grad()
def eval_model(model, out_csv):
    rows, i0, t0 = [], 0, time.time()
    for high, low in LOADER:
        high, low = high.to(dev, non_blocking=True), low.to(dev, non_blocking=True)
        pred = model(low, side_loss=False).clamp(0,1)[:, :, :high.shape[2], :high.shape[3]]
        mse  = ((pred-high)**2).mean(dim=(1,2,3))
        psnr = 20*torch.log10(1.0/(mse.sqrt()+1e-8))
        ssim = _ssim(pred, high, data_range=1.0, size_average=False)
        lp   = LPIPS_NET(pred*2-1, high*2-1).flatten()          # documented [-1,1] scaling
        lpdk = LPIPS_NET(pred,      high     ).flatten()        # DarkIR's unscaled convention
        for j in range(high.shape[0]):
            rows.append((i0+j, FNAMES[i0+j], float(psnr[j]), float(ssim[j]),
                         float(lp[j]), float(lpdk[j])))
        i0 += high.shape[0]
        if i0 % 400 < BATCH:
            r = (time.time()-t0)/max(i0,1)
            print(f'    {i0}/{len(DS)}  ~{r*(len(DS)-i0)/60:.1f} min left', flush=True)
    with open(out_csv,'w',newline='') as f:
        w = csv.writer(f); w.writerow(['idx','fname','psnr','ssim','lpips','lpips_darkirconv'])
        w.writerows(rows)
    a = np.array([r[2:] for r in rows])
    return a.mean(0), len(rows)

## 5. RUN — 12 evals, idempotent

In [ ]:
for name, fn in TARGETS:
    out = os.path.join(PERIMG, f'per_image_{name}.csv')
    if os.path.exists(out):
        print(f'✅ {name}: exists — skip'); continue
    p = find_ckpt(fn)
    if not p:
        print(f'⏭  {name}: checkpoint {fn} not attached — skip'); continue
    print(f'\n=== EVAL {name}  ({os.path.relpath(p, INPUT_BASE)}) ===', flush=True)
    model, ck = load_model(p)
    (ps, ss, lp, lpdk), n = eval_model(model, out)
    print(f'  → PSNR {ps:.4f}  SSIM {ss:.4f}  LPIPS {lp:.4f}  LPIPS(DarkIR-conv) {lpdk:.4f}  n={n}')
    # curated EMA-only copy for future sessions (~13 MB vs ~150 MB)
    sp = os.path.join(STRIP, os.path.basename(p))
    if not os.path.exists(sp):
        st = (ck.get('ema') or ck.get('model') or ck.get('params')) if isinstance(ck, dict) else ck
        torch.save({'ema': {k.replace('module.',''):v for k,v in st.items()},
                    'epoch': ck.get('epoch') if isinstance(ck, dict) else None}, sp)
    del model; torch.cuda.empty_cache()

# alias copies so existing notebooks/table code find the names they expect
for src_name, alist in ALIASES.items():
    s = os.path.join(PERIMG, f'per_image_{src_name}.csv')
    if os.path.exists(s):
        for a in alist:
            d = os.path.join(PERIMG, f'per_image_{a}.csv')
            if not os.path.exists(d): shutil.copy2(s, d); print('alias:', os.path.basename(d))
print('\nRUN loop complete.')

## 6. Verify pairing is now real + bundle

In [ ]:
# ── The check that would have caught the original bug ────────────────────────
import csv
def load(name):
    p = os.path.join(PERIMG, f'per_image_{name}.csv')
    if not os.path.exists(p): return None
    rows = list(csv.DictReader(open(p)))
    return ([r['fname'] for r in rows], np.array([float(r['psnr']) for r in rows]))

pairs = [('stage4_physir_final','stage5_baseline'), ('stage4_physir_final','darkir_m_released'),
         ('abl_full_last','abl_baseline_last'), ('abl_full_last','abl_percep_last')]
print(f'{"pair":52s} {"fname match":>12s} {"corr":>7s}  verdict')
for a,b in pairs:
    A, B = load(a), load(b)
    if not (A and B): print(f'{a+" vs "+b:52s}  (not both present)'); continue
    same = A[0] == B[0]
    r = float(np.corrcoef(A[1], B[1])[0,1])
    v = 'PAIRED ✅' if same and r > 0.4 else 'PROBLEM ❌'
    print(f'{a+" vs "+b:52s} {str(same):>12s} {r:7.3f}  {v}')
print('\nExpect fname match = True and corr > 0.4 (image difficulty dominates and now cancels).')
print('Before the fix these correlations were ~0.0 — that was the tell.')

# ── Packaging: must NEVER raise. A crash here after hours of eval strands the run,
# ── which is exactly what happened on the 2026-08-14 session.
import glob as _g

# 1. Sweep the whole working tree — if any CSV landed outside PERIMG, find it rather
#    than silently reporting an empty result set.
_found = sorted(_g.glob(os.path.join(WORK, '**', 'per_image_*.csv'), recursive=True))
_stray = [f for f in _found if os.path.dirname(f) != PERIMG]
print(f'per_image_*.csv found under {WORK}: {len(_found)}  (outside per_image/: {len(_stray)})')
for f in _stray:
    print('   stray →', f)
    os.makedirs(PERIMG, exist_ok=True)
    shutil.copy2(f, os.path.join(PERIMG, os.path.basename(f)))

os.makedirs(PERIMG, exist_ok=True)          # recreate if something removed it
_csvs = sorted(f for f in os.listdir(PERIMG) if f.endswith('.csv'))
print(f'\nper_image/ holds {len(_csvs)} csv(s):')
for f in _csvs:
    print(f'   {f}  ({sum(1 for _ in open(os.path.join(PERIMG,f)))-1} rows)')

if _csvs:
    try:
        z = shutil.make_archive(os.path.join(WORK, 'per_image_corrected'), 'zip', WORK, 'per_image')
        print(f'\n✅ archived → {z}  ({os.path.getsize(z)/2**20:.1f} MB)')
    except Exception as e:
        # Fall back to writing the zip explicitly; the CSVs themselves are already
        # saved in the notebook output, so nothing is lost either way.
        print(f'\n⚠ make_archive failed ({type(e).__name__}: {e}) — writing zip directly')
        import zipfile
        zp = os.path.join(WORK, 'per_image_corrected.zip')
        with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
            for f in _csvs:
                zf.write(os.path.join(PERIMG, f), os.path.join('per_image', f))
        print(f'✅ archived → {zp}  ({os.path.getsize(zp)/2**20:.1f} MB)')
else:
    print('\n❌ NO per-image CSVs were produced — check the In[3] status table above: '
          'every target likely reported "CKPT MISSING". Nothing to archive.')

print('\nstripped ckpts:', len(os.listdir(STRIP)) if os.path.isdir(STRIP) else 0, 'files')
print('\nSave a version, then attach this notebook\'s output to the figures/tables notebook')
print('(name it with "corrected" in the slug so it takes precedence).')